# 01 — Data cleaning: FAERS GLP-1 reports
**Owner:** Rutu  ·  **Output:** the shared files everyone else uses

| Step | What happens |
|---|---|
| 0 | Setup and imports |
| 1 | Find the raw FAERS files (any quarters that are downloaded) |
| 2 | Look at one raw file to see the mess |
| 3 | Process each quarter: keep only GLP-1 primary-suspect reports |
| 4 | Remove duplicate case versions and deleted cases |
| 5 | Clean the columns (age, weight, sex, reporter, outcomes) |
| 6 | Quality checks |
| 7 | Save the shared files + pipeline counts |
| 8 | Quick summary |

All rules (drug list, serious codes, unit conversions) come from `src/definitions.py`.

## Step 0 — Setup

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.definitions import *
from src import faers_pipeline as fp

RAW_DIR = PROJECT_ROOT / "data" / "01_raw"
OUT_DIR = PROJECT_ROOT / "data" / "02_intermediate"
OUT_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", 50)
print("Project root:", PROJECT_ROOT)

Project root: /Users/rutujasaste/data230-group4-project


## Step 1 — Find the raw files
Each FAERS quarter has 7 tables. We use 5 of them: **DEMO** (patient), **DRUG** (medicines),
**REAC** (side effects), **OUTC** (outcomes) and **INDI** (why the drug was taken).
The search works whatever folder layout the download script used.

In [ ]:
quarters = fp.find_quarter_files(RAW_DIR)
deleted_files = fp.find_deleted_files(RAW_DIR)

for q, files in quarters.items():
    missing = [t for t in fp.TABLES if t not in files]
    print(q, "->", sorted(files), "MISSING: " + str(missing) if missing else "")
print(f"\n{len(quarters)} quarters found, {len(deleted_files)} 'Deleted' file(s) found")
assert quarters, "No FAERS files found: run `python src/ingest.py` first"

2022Q1 -> ['DEMO', 'DRUG', 'INDI', 'OUTC', 'REAC'] 
2022Q2 -> ['DEMO', 'DRUG', 'INDI', 'OUTC', 'REAC'] 
2022Q3 -> ['DEMO', 'DRUG', 'INDI', 'OUTC', 'REAC'] 
2022Q4 -> ['DEMO', 'DRUG', 'INDI', 'OUTC', 'REAC'] 
2023Q1 -> ['DEMO', 'DRUG', 'INDI', 'OUTC', 'REAC'] 
2023Q2 -> ['DEMO', 'DRUG', 'INDI', 'OUTC', 'REAC'] 
2023Q3 -> ['DEMO', 'DRUG', 'INDI', 'OUTC', 'REAC'] 
2023Q4 -> ['DEMO', 'DRUG', 'INDI', 'OUTC', 'REAC'] 
2024Q1 -> ['DEMO', 'DRUG', 'INDI', 'OUTC', 'REAC'] 
2024Q2 -> ['DEMO', 'DRUG', 'INDI', 'OUTC', 'REAC'] 
2024Q3 -> ['DEMO', 'DRUG', 'INDI', 'OUTC', 'REAC'] 
2024Q4 -> ['DEMO', 'DRUG', 'INDI', 'OUTC', 'REAC'] 
2025Q1 -> ['DEMO', 'DRUG', 'INDI', 'OUTC', 'REAC'] 
2025Q2 -> ['DEMO', 'DRUG', 'INDI', 'OUTC', 'REAC'] 
2025Q3 -> ['DEMO', 'DRUG', 'INDI', 'OUTC', 'REAC'] 
2025Q4 -> ['DEMO', 'DRUG', 'INDI', 'OUTC', 'REAC'] 
2026Q1 -> ['DEMO', 'DRUG', 'INDI', 'OUTC', 'REAC'] 
2026Q2 -> ['DEMO', 'DRUG', 'INDI', 'OUTC', 'REAC'] 

18 quarters found, 18 'Deleted' file(s) found


## Step 2 — Look at one raw file
This shows why cleaning is needed: ages in different units (`age_cod`), weights in KG or LBS,
blank fields, and one case appearing with several `caseversion` numbers.

In [ ]:
first_q = next(iter(quarters))
demo_sample = fp.read_faers(quarters[first_q]["DEMO"])
print(first_q, "DEMO:", demo_sample.shape[0], "rows x", demo_sample.shape[1], "columns")
print("\nAge units:\n", demo_sample["age_cod"].value_counts(dropna=False).head(8))
print("\nWeight units:\n", demo_sample["wt_cod"].value_counts(dropna=False).head(5))
demo_sample.head()

2022Q1 DEMO: 461623 rows x 25 columns

Age units:
 age_cod
YR     246339
NaN    209227
DEC      2888
DY       1646
MON      1314
WK        197
HR         12
Name: count, dtype: int64

Weight units:
 wt_cod
NaN    387498
KG      73319
LBS       806
Name: count, dtype: int64


,primaryid,caseid,caseversion,i_f_code,event_dt,mfr_dt,init_fda_dt,fda_dt,rept_cod,auth_num,mfr_num,mfr_sndr,lit_ref,age,age_cod,age_grp,sex,e_sub,wt,wt_cod,rept_dt,to_mfr,occp_cod,reporter_country,occr_country
0,1000597210,10005972,10,F,20140221,20220202,20140313,20220206,EXP,NaN,PHHO2014DE003546,NOVARTIS,NaN,66,YR,NaN,F,Y,74,KG,20220203,NaN,MD,DE,DE
1,1001443244,10014432,44,F,NaN,20220216,20140317,20220225,EXP,NaN,CA-009507513-1403CAN006934,MERCK,NaN,64,YR,NaN,F,Y,69,KG,20220225,NaN,HP,CA,CA
2,100144836,10014483,6,F,NaN,20220106,20140317,20220111,EXP,NaN,CA-009507513-1403CAN003893,MERCK,"Jaworsky D, Thompson C, Yudin MH, Bitnun A, Br...",8,WK,NaN,NaN,Y,NaN,NaN,20220111,NaN,HP,CA,CA
3,1001678110,10016781,10,F,20120330,20211229,20140318,20220106,EXP,NaN,PHHY2012CA028320,NOVARTIS,NaN,45,YR,NaN,F,Y,NaN,NaN,20220106,NaN,HP,CA,CA
4,1002221566,10022215,66,F,20110101,20220216,20140319,20220222,EXP,NaN,PHHY2011CA101736,NOVARTIS,NaN,NaN,NaN,NaN,F,Y,NaN,NaN,20220222,NaN,CN,CA,CA


## Step 3 — Process each quarter
For every quarter we:
1. Read DRUG and find rows that mention a GLP-1 drug (by ingredient or brand name).
2. Leave out insulin combination products (Xultophy, Soliqua).
3. Keep reports where the GLP-1 is the **primary suspect** (`role_cod == "PS"`).
4. Attach that report's patient info (DEMO), outcomes (OUTC), reactions (REAC) and indication (INDI).

We go **one quarter at a time** because DRUG and REAC have millions of rows each;
only the small GLP-1 part is kept in memory.

In [ ]:
reports_list, reactions_list, demo_light_list, counts_list = [], [], [], []

for q, files in quarters.items():
    rep, reac, demo_light, counts = fp.process_quarter(q, files)
    reports_list.append(rep)
    reactions_list.append(reac)
    demo_light_list.append(demo_light)
    counts_list.append(counts)
    print(f"{q}: {counts['raw_reports']:>9,} reports | GLP-1 mentioned {counts['glp1_mentioned_reports']:>7,} "
          f"| GLP-1 primary suspect {counts['glp1_ps_reports']:>7,}")

reports_all = pd.concat(reports_list, ignore_index=True)
reactions_all = pd.concat(reactions_list, ignore_index=True)
demo_light_all = pd.concat(demo_light_list, ignore_index=True)
print("\nGLP-1 primary-suspect reports before de-duplication:", len(reports_all))

2022Q1:   461,623 reports | GLP-1 mentioned   6,635 | GLP-1 primary suspect   5,368
2022Q2:   435,618 reports | GLP-1 mentioned   5,764 | GLP-1 primary suspect   4,409
2022Q3:   446,511 reports | GLP-1 mentioned   7,556 | GLP-1 primary suspect   5,956
2022Q4:   483,643 reports | GLP-1 mentioned   7,762 | GLP-1 primary suspect   6,052
2023Q1:   432,144 reports | GLP-1 mentioned  12,202 | GLP-1 primary suspect  10,389
2023Q2:   418,592 reports | GLP-1 mentioned  10,386 | GLP-1 primary suspect   8,445
2023Q3:   407,522 reports | GLP-1 mentioned  13,924 | GLP-1 primary suspect  11,884
2023Q4:   415,379 reports | GLP-1 mentioned  10,012 | GLP-1 primary suspect   7,772
2024Q1:   406,184 reports | GLP-1 mentioned  21,742 | GLP-1 primary suspect  19,302
2024Q2:   397,119 reports | GLP-1 mentioned  13,349 | GLP-1 primary suspect  10,713
2024Q3:   405,513 reports | GLP-1 mentioned  19,676 | GLP-1 primary suspect  16,965
2024Q4:   410,849 reports | GLP-1 mentioned  15,925 | GLP-1 primary suspect 

## Step 4 — Remove duplicates and deleted cases
A **case** (`caseid`) can be sent several times as it is updated; each update gets a new
`caseversion` and a new `primaryid`, sometimes in a later quarter.
We keep only the **latest version** of each case, across all quarters.
FDA also lists cases that were withdrawn in a *Deleted* file; those are removed too.

In [ ]:
deleted_caseids = fp.read_deleted_caseids(deleted_files)
keep_ids, n_unique_cases = fp.latest_primaryids(demo_light_all, deleted_caseids)

mentioned_ids = set().union(*[c["glp1_mentioned_ids"] for c in counts_list])
before = len(reports_all)
reports_latest = reports_all[reports_all["primaryid"].isin(keep_ids)].copy()
reactions_latest = reactions_all[reactions_all["primaryid"].isin(keep_ids)].copy()

# Same report can appear in two quarterly files -> keep one copy (the latest quarter)
dups = reports_latest["primaryid"].duplicated().sum()
reports_latest = (reports_latest.sort_values("year_quarter")
                  .drop_duplicates("primaryid", keep="last"))
reactions_latest = reactions_latest.drop_duplicates(["primaryid", "pt"])
print("Duplicate report rows removed:", dups)

print("Deleted case ids listed by FDA:", len(deleted_caseids))
print(f"GLP-1 PS reports: {before:,} -> {len(reports_latest):,} after keeping the latest version")

Duplicate report rows removed: 4
Deleted case ids listed by FDA: 87628
GLP-1 PS reports: 269,663 -> 252,568 after keeping the latest version


## Step 5 — Clean the columns
| Column | Rule (from `src/definitions.py`) |
|---|---|
| `age_years` | Converted from YR / DEC / MON / WK / DY / HR; outside 0–110 → missing |
| `age_group` | Under 18, 18–39, 40–64, 65 and over, Unknown |
| `weight_kg` | KG kept, LBS × 0.4536; outside 30–300 → missing |
| `sex` | M, F, otherwise Unknown |
| `reporter_type`, `is_hcp` | From `occp_cod` |
| `indication_group` | Diabetes, Weight loss, Other, Unknown |
| `is_serious` | 1 if any outcome is DE, LT, HO, DS, CA or RI |

Missing values are **kept and labelled**, never dropped silently.

In [ ]:
glp1_reports = fp.clean_reports(reports_latest)

# Reaction table: add drug and target so it can be used on its own
glp1_reactions = reactions_latest.merge(
    glp1_reports[["primaryid", "drug", "is_serious"]], on="primaryid", how="inner"
)[["primaryid", "drug", "pt", "is_serious"]]

print("glp1_reports:", glp1_reports.shape)
print("glp1_reactions:", glp1_reactions.shape)
glp1_reports.head()

glp1_reports: (252568, 31)
glp1_reactions: (590454, 4)


,primaryid,caseid,year,quarter,year_quarter,drug,brand,is_compounded,drugname,prod_ai,route,dose_amt,dose_unit,dose_form,age_raw,age_cod,age_years,age_group,weight_kg,sex,reporter_type,is_hcp,reporter_country,indication_group,n_reactions,n_other_drugs,is_death,is_life_threatening,is_hospitalized,is_disability,is_serious
0,1300919515,13009195,2022,1,2022Q1,liraglutide,Victoza,0,VICTOZA,LIRAGLUTIDE,Subcutaneous,NaN,NaN,Solution for injection,57.0,YR,57.0,40-64,120.2,M,Physician,1,US,Diabetes,17,41,0,0,1,1,1
3508,205189461,20518946,2022,1,2022Q1,liraglutide,Victoza,0,VICTOZA,LIRAGLUTIDE,Subcutaneous,1.2,MG,Solution for injection,80.0,YR,80.0,65 and over,NaN,F,Consumer,0,US,Unknown,4,2,0,0,0,0,0
3507,205189451,20518945,2022,1,2022Q1,liraglutide,Victoza,0,VICTOZA,LIRAGLUTIDE,Subcutaneous,NaN,NaN,Solution for injection,65.0,YR,65.0,65 and over,NaN,F,Physician,1,US,Diabetes,3,0,0,0,0,0,0
3506,205189441,20518944,2022,1,2022Q1,liraglutide,Victoza,0,VICTOZA,LIRAGLUTIDE,Subcutaneous,1.8,MG,Solution for injection,NaN,NaN,NaN,Unknown,NaN,F,Consumer,0,US,Diabetes,1,0,0,0,0,0,0
3505,205189311,20518931,2022,1,2022Q1,liraglutide,Victoza,0,VICTOZA,LIRAGLUTIDE,Subcutaneous,NaN,NaN,Solution for injection,NaN,NaN,NaN,Unknown,NaN,F,Consumer,0,US,Diabetes,3,8,0,0,0,0,0


## Step 6 — Quality checks (every line must pass)

In [ ]:
assert glp1_reports["primaryid"].is_unique, "one row per report"
assert glp1_reports["caseid"].is_unique, "one report per case (latest version)"
assert set(glp1_reports["drug"]) <= set(GLP1_DRUGS), "only the 4 GLP-1 drugs"
assert set(glp1_reports["is_serious"]) <= {0, 1}
assert glp1_reports["age_years"].dropna().between(AGE_MIN, AGE_MAX).all()
assert glp1_reports["weight_kg"].dropna().between(WEIGHT_MIN, WEIGHT_MAX).all()
assert set(glp1_reactions["primaryid"]) <= set(glp1_reports["primaryid"])

missing_pct = (glp1_reports[["age_years", "weight_kg", "dose_amt", "route"]].isna().mean() * 100).round(1)
print("All checks passed.\n\n% missing in key columns:")
print(missing_pct.to_string())
print("Sex = Unknown:", round((glp1_reports["sex"] == "Unknown").mean() * 100, 1), "%")
print("Indication = Unknown:", round((glp1_reports["indication_group"] == "Unknown").mean() * 100, 1), "%")

All checks passed.

% missing in key columns:
age_years    38.6
weight_kg    92.1
dose_amt     37.4
route        24.3
Sex = Unknown: 16.4 %
Indication = Unknown: 52.6 %


## Step 7 — Save the shared files
* **Parquet** for Python (small, fast), **CSV** for Tableau.
* While only one year is downloaded, the files get that year as a suffix (`glp1_reports_2025`);
  with several years they are saved without a suffix (`glp1_reports`).
* `pipeline_counts.csv` feeds the funnel chart C1.

In [ ]:
years = sorted({q[:4] for q in quarters})
suffix = f"_{years[0]}" if len(years) == 1 else ""

glp1_reports.to_parquet(OUT_DIR / f"glp1_reports{suffix}.parquet", index=False)
glp1_reports.to_csv(OUT_DIR / f"glp1_reports{suffix}.csv", index=False)
glp1_reactions.to_parquet(OUT_DIR / f"glp1_reactions{suffix}.parquet", index=False)
glp1_reactions.to_csv(OUT_DIR / f"glp1_reactions{suffix}.csv", index=False)

pipeline_counts = pd.DataFrame([
    {"step": "1. Raw reports (all drugs)", "reports": sum(c["raw_reports"] for c in counts_list)},
    {"step": "2. Unique cases (latest version, deleted removed)", "reports": n_unique_cases},
    {"step": "3. Mention a GLP-1 drug", "reports": len(mentioned_ids & keep_ids)},
    {"step": "4. GLP-1 is the primary suspect (final)", "reports": len(glp1_reports)},
])
pipeline_counts.to_csv(OUT_DIR / f"pipeline_counts{suffix}.csv", index=False)

for f in sorted(OUT_DIR.glob(f"*{suffix}.*")):
    print(f"{f.name:40s} {f.stat().st_size / 1e6:8.2f} MB")
pipeline_counts

glp1_reactions.csv                          24.76 MB
glp1_reactions.parquet                       3.06 MB
glp1_reactions_2025.csv                      8.54 MB
glp1_reactions_2025.parquet                  1.25 MB
glp1_reports.csv                            44.40 MB
glp1_reports.parquet                         4.70 MB
glp1_reports_2025.csv                       15.54 MB
glp1_reports_2025.parquet                    1.87 MB
pipeline_counts.csv                          0.00 MB
pipeline_counts_2025.csv                     0.00 MB


,step,reports
0,1. Raw reports (all drugs),7557824
1,"2. Unique cases (latest version, deleted removed)",6433927
2,3. Mention a GLP-1 drug,290435
3,4. GLP-1 is the primary suspect (final),252568


## Step 8 — Quick summary (sanity check, not a slide chart)

In [ ]:
summary = (glp1_reports.groupby("drug")
           .agg(reports=("primaryid", "count"),
                pct_serious=("is_serious", "mean"),
                median_age=("age_years", "median"),
                pct_female=("sex", lambda s: (s == "F").mean()))
           .sort_values("reports", ascending=False))
summary["pct_serious"] = (summary["pct_serious"] * 100).round(1)
summary["pct_female"] = (summary["pct_female"] * 100).round(1)
print("Overall % serious:", round(glp1_reports["is_serious"].mean() * 100, 1))
summary

Overall % serious: 11.3


,reports,pct_serious,median_age,pct_female
drug,,,,
tirzepatide,151590,6.8,55.0,60.9
semaglutide,67110,20.6,60.0,62.1
dulaglutide,28403,10.4,63.0,52.3
liraglutide,5465,26.5,55.0,68.9
